In [43]:
import os
import csv
import chromadb
from dotenv import load_dotenv
from google import genai
import pandas as pd
from chromadb import Documents, EmbeddingFunction, Embeddings
from IPython.display import Markdown



In [44]:
load_dotenv()
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

In [45]:
for m in client.models.list():
  if 'embedContent' in m.supported_actions:
    print(m.name)

models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp


In [9]:
# TODO: https://github.com/google-gemini/cookbook/blob/main/examples/chromadb/Vectordb_with_chroma.ipynb

In [51]:
DOCUMENT1 = """
  Operating the Climate Control System  Your Googlecar has a climate control
  system that allows you to adjust the temperature and airflow in the car.
  To operate the climate control system, use the buttons and knobs located on
  the center console.  Temperature: The temperature knob controls the
  temperature inside the car. Turn the knob clockwise to increase the
  temperature or counterclockwise to decrease the temperature.
  Airflow: The airflow knob controls the amount of airflow inside the car.
  Turn the knob clockwise to increase the airflow or counterclockwise to
  decrease the airflow. Fan speed: The fan speed knob controls the speed
  of the fan. Turn the knob clockwise to increase the fan speed or
  counterclockwise to decrease the fan speed.
  Mode: The mode button allows you to select the desired mode. The available
  modes are: Auto: The car will automatically adjust the temperature and
  airflow to maintain a comfortable level.
  Cool: The car will blow cool air into the car.
  Heat: The car will blow warm air into the car.
  Defrost: The car will blow warm air onto the windshield to defrost it.
"""
DOCUMENT2 = """
  Your Googlecar has a large touchscreen display that provides access to a
  variety of features, including navigation, entertainment, and climate
  control. To use the touchscreen display, simply touch the desired icon.
  For example, you can touch the \"Navigation\" icon to get directions to
  your destination or touch the \"Music\" icon to play your favorite songs.
"""
DOCUMENT3 = """
  Shifting Gears Your Googlecar has an automatic transmission. To
  shift gears, simply move the shift lever to the desired position.
  Park: This position is used when you are parked. The wheels are locked
  and the car cannot move.
  Reverse: This position is used to back up.
  Neutral: This position is used when you are stopped at a light or in traffic.
  The car is not in gear and will not move unless you press the gas pedal.
  Drive: This position is used to drive forward.
  Low: This position is used for driving in snow or other slippery conditions.
"""

# documents = [DOCUMENT1, DOCUMENT2, DOCUMENT3]

In [99]:
documents = []
with open('output.csv', 'r', encoding='utf-8') as file:
    loaded_csv = csv.reader(file)
    header = next(loaded_csv)


    for row in loaded_csv:
        time_stamps = row[0]
        doc = row[1]+"\n"+"Segment Subtitle"+row[3]+"\n"+"Segment keywords:"+row[2].strip('[]')
        documents.append((time_stamps, doc))

In [100]:
documents[:5]

[('0s-10s',
  "This segment presents a stark visual of a completely black screen, punctuated by a slowly rotating, orange circle. The accompanying text introduces the fundamental concept of a vector in linear algebra, stating that it's the ‘root-of-it-all building block.’ The visual simplicity and the initial definition establish a foundational element for the video's subsequent explanation.\nSegment SubtitleThe fundamental, root-of-it-all building block for linear algebra is the vector.\nSegment keywords:'vector', 'linear algebra', 'black screen', 'circle', 'fundamentals'"),
 ('10s-20s',
  "The segment continues with a black screen and the phrase ‘The fundamental, root-of-it-all building block for linear algebra is the vector.’ This reinforces the initial concept.  The subsequent visuals show a vector represented as an arrow on a grid, highlighting its length and direction. The grid provides a visual context for understanding the vector's position in space.\nSegment SubtitleThe fundam

In [101]:
from google.genai import types

class GeminiEmbeddingFunction(EmbeddingFunction):
  def __call__(self, input: Documents) -> Embeddings:
    EMBEDDING_MODEL_ID = "models/embedding-001"  # @param ["models/embedding-001", "models/text-embedding-004", "models/gemini-embedding-exp-03-07", "models/gemini-embedding-exp"] {"allow-input": true, "isTemplate": true}
    title = "Custom query"
    response = client.models.embed_content(
        model=EMBEDDING_MODEL_ID,
        contents=input,
        config=types.EmbedContentConfig(
          task_type="retrieval_document",
          title=title
        )
    )

    return [emb.values for emb in response.embeddings]

In [104]:
def create_chroma_db(documents, name):
  chroma_client = chromadb.PersistentClient(path='./chromadb')
  db = chroma_client.get_or_create_collection(
      name=name,
      embedding_function=GeminiEmbeddingFunction()
  )

  for i, d in enumerate(documents):
    db.add(
      documents=d[1],
      ids=str(i),
      metadatas={'timestamps':d[0]}
    )
  return db

In [105]:
db = create_chroma_db(documents, "3b1b_vector")


/tmp/ipykernel_4973/108476745.py:5: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  embedding_function=GeminiEmbeddingFunction()


In [106]:
sample_data = db.get(include=['documents', 'embeddings'])

df = pd.DataFrame({
    "IDs": sample_data['ids'][:3],
    "Documents": sample_data['documents'][:3],
    "Embeddings": [str(emb)[:50] + "..." for emb in sample_data['embeddings'][:3]]  # Truncate embeddings
})

print(df)

  IDs                                          Documents  \
0   0  This segment presents a stark visual of a comp...   
1   1  The segment continues with a black screen and ...   
2   2  This segment introduces the three perspectives...   

                                          Embeddings  
0  [ 4.31862846e-02 -6.79173768e-02 -7.30723329e-...  
1  [ 4.86326823e-03 -4.40525115e-02  5.06896386e-...  
2  [-2.02238038e-02 -5.14030829e-02  5.28389867e-...  


In [125]:
def get_relevant_passage(query, db):
  ret_seg_data = []
  retrieved_data = db.query(query_texts=[query], n_results=10)
  ret_ids = retrieved_data['ids'][0] 
  ret_data = db.get(ids=ret_ids)
  ts = ret_data['metadatas']
  docs = ret_data['documents']

  for i in range(10):
    chunk = {"time_stamps":ts[i]['timestamps'], "Segment content": docs[i] }
    ret_seg_data.append(chunk)
  

  
  return "".join(ret_seg_data)

In [126]:
# Perform embedding search
passage = get_relevant_passage("vector addition", db)
passage

TypeError: sequence item 0: expected str instance, dict found

In [129]:

def make_prompt(query, relevant_passage):
  prompt = f"""
    You are a helpful and informative video chunk/segment finder bot that find 
    relevant video chunk/segment description related to the QUESTION from the CHUNK LIST.
    Then output the relevant chunks with time_stamps in array like
    [20s-50s, 50s-120s, ...].
    If the chunk/segemnts are irrelevant then you may ignore it.
    QUESTION: {query}
    CHUNK LIST: {relevant_passage}

    ANSWER:
  """

  return prompt

In [130]:
query = "Which sections are about vector addition?"
prompt = make_prompt(query, passage)
prompt

'\n    You are a helpful and informative video chunk/segment finder bot that find \n    relevant video chunk/segment description related to the QUESTION from the CHUNK LIST.\n    Then output the relevant chunks with time_stamps in array like\n    [20s-50s, 50s-120s, ...].\n    If the chunk/segemnts are irrelevant then you may ignore it.\n    QUESTION: Which sections are about vector addition?\n    CHUNK LIST: [{\'time_stamps\': \'280s-290s\', \'Segment content\': "This segment illustrates vector addition through a visual example. Two vectors, one pointing upwards and slightly to the right, and the other pointing to the right and downwards, are shown. The text explains that to add these vectors, the tail of the second vector is placed at the tip of the first, and the resulting vector is drawn from the first vector\'s tail to the second vector\'s tip.\\nSegment SubtitleAlright, so back to vector addition and multiplication by numbers.\\nAfter all, every topic in linear algebra is going t

In [131]:
MODEL_ID = "gemma-3-27b-it"  # @param ["gemini-2.5-flash-lite-preview-06-17", "gemini-2.5-flash", "gemini-2.5-flash","gemini-2.5-pro"] {"allow-input": true, "isTemplate": true}
answer = client.models.generate_content(
    model = MODEL_ID,
    contents = prompt
)
Markdown(answer.text)

```json
[
  "280s-290s",
  "290s-300s",
  "300s-310s",
  "310s-320s",
  "320s-330s",
  "370s-380s",
  "380s-390s",
  "390s-400s",
  "400s-410s"
]
```